# Comprehensive Survival Analysis Benchmark

**Bachelor Thesis: Application of ML Models in Survival Analysis**

This notebook showcases the full experimental pipeline:

1. **Synthetic Data Generation** — 5 setups + interpretability scenario
2. **Model Overview** — 6 survival models (classical + deep learning)
3. **Quick Benchmark** — BenchmarkRunner with default hyperparameters
4. **Full Evaluation** — Nested 5×5 cross-validation with HP tuning
5. **Per-Setup Deep Dives** — Setup-specific analyses
6. **Cross-Setup Summary** — Learning curves and overall rankings
7. **Interpretability** — SurvLIME + SurvSHAP(t) on known ground-truth
8. **Conclusions** — Key findings and summary figure

### Configuration

- `QUICK_MODE = True`: Runs a reduced experiment for fast iteration (2 setups × 3 models for nested CV, Setup 5 subsampled).
- `QUICK_MODE = False`: Full experiment — all 5 setups × 6 models, full nested CV. Expect ~2–3 hours.
- `RETRAIN_MODELS = True`: Train all models from scratch and save checkpoints.
- `RETRAIN_MODELS = False`: Load previously saved models from `notebooks/saved_models/` (must have been trained before).

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from lifelines import KaplanMeierFitter, CoxPHFitter, NelsonAalenFitter
from sksurv.metrics import brier_score, concordance_index_ipcw
from sksurv.util import Surv
from sklearn.model_selection import train_test_split

from src.data.generate import (
    generate_setup_1, generate_setup_2, generate_setup_3,
    generate_setup_4, generate_setup_5, generate_setup_interp,
)
from src.data.types import SurvivalData
from src.models import ALL_MODELS
from src.models.base import SurvivalModel, CauseSpecificWrapper
from src.benchmark.runner import BenchmarkRunner
from src.benchmark.nested_cv_runner import NestedCVRunner
from src.evaluation.metrics import (
    evaluate_model, concordance_index, integrated_brier_score,
    time_dependent_auc, calibration_curve,
)
from src.evaluation.visualizations import (
    plot_metric_heatmap, plot_rank_chart,
    plot_brier_score_over_time, plot_calibration_comparison,
)

sns.set_style('whitegrid')
%matplotlib inline

SEED = 42
np.random.seed(SEED)

# ── Paths (absolute, robust) ──
PROJECT_ROOT = Path(os.path.abspath('..')).resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
SAVED_MODELS_DIR = Path('saved_models').resolve()
RESULTS_DIR.mkdir(exist_ok=True)
SAVED_MODELS_DIR.mkdir(exist_ok=True)

print('All imports loaded successfully.')
print(f'Results dir:      {RESULTS_DIR}')
print(f'Saved models dir: {SAVED_MODELS_DIR}')

In [ ]:
# ── Configuration ──
QUICK_MODE = True        # Set False for full experiment (much longer runtime)
RETRAIN_MODELS = True    # Set False to load saved models from saved_models/

SETUP_5_SUBSAMPLE = 5000  # Subsample Setup 5 from 50k in quick mode

if QUICK_MODE:
    print('QUICK_MODE enabled:')
    print(f'  - Setup 5 subsampled to {SETUP_5_SUBSAMPLE}')
    print('  - NestedCV limited to 2 setups x 3 models')
else:
    print('Full experiment mode — expect ~2-3 hours runtime')

if RETRAIN_MODELS:
    print('RETRAIN_MODELS = True — all models will be trained from scratch and saved.')
else:
    print(f'RETRAIN_MODELS = False — loading models from {SAVED_MODELS_DIR}/')

# ── Helper: save / load a model ──
def save_model(model, name: str, tag: str = '') -> Path:
    """Save a fitted model to SAVED_MODELS_DIR/{tag}/{name}.pkl."""
    subdir = SAVED_MODELS_DIR / tag if tag else SAVED_MODELS_DIR
    path = subdir / f'{name}.pkl'
    model.save(path)
    return path

def load_model(name: str, tag: str = '') -> SurvivalModel:
    """Load a model from SAVED_MODELS_DIR/{tag}/{name}.pkl."""
    subdir = SAVED_MODELS_DIR / tag if tag else SAVED_MODELS_DIR
    path = subdir / f'{name}.pkl'
    if not path.exists():
        raise FileNotFoundError(f'No saved model at {path}. Set RETRAIN_MODELS=True first.')
    return SurvivalModel.load(path)

def model_exists(name: str, tag: str = '') -> bool:
    """Check if a saved model exists."""
    subdir = SAVED_MODELS_DIR / tag if tag else SAVED_MODELS_DIR
    return (subdir / f'{name}.pkl').exists()

---
## 1. Synthetic Data Generation

We generate 5 synthetic setups designed to stress-test different model classes, plus a dedicated interpretability scenario:

| Setup | DGM | n | p | Challenge | Expected winner |
|-------|-----|---|---|-----------|----------------|
| 1 — Clean PH | Weibull PH | 1000 | 10 | Sanity check | CoxPH |
| 2 — Non-PH | Weibull, β(t) | 1500 | 20 | Time-varying effects | RSF / Deep |
| 3 — High-dim | Weibull PH, sparse | 500 | 2000 | p >> n | CoxNet / Deep |
| 4 — Competing risks | Multi-state Weibull | 2000 | 15 | Multiple causes | DeepHit / SurvTRACE |
| 5 — Large-n | Gompertz | 50000 | 30 | Scale + heavy censoring | RSF / Deep |
| Interp | Weibull, mixed effects | 1000 | 12 | Known ground truth | — |

In [ ]:
t0 = time.perf_counter()

data1 = generate_setup_1(seed=SEED)
data2 = generate_setup_2(seed=SEED)
data3 = generate_setup_3(seed=SEED)
data4 = generate_setup_4(seed=SEED)
data5_full = generate_setup_5(seed=SEED)
data_interp = generate_setup_interp(seed=SEED)

# Subsample Setup 5 in quick mode
if QUICK_MODE:
    rng = np.random.default_rng(SEED)
    idx = rng.choice(data5_full.X.shape[0], SETUP_5_SUBSAMPLE, replace=False)
    data5 = SurvivalData(
        X=data5_full.X[idx], T=data5_full.T[idx], E=data5_full.E[idx],
        feature_names=data5_full.feature_names,
        true_event_times=data5_full.true_event_times[idx],
        true_betas=data5_full.true_betas,
        setup_name=data5_full.setup_name,
        config=data5_full.config,
    )
else:
    data5 = data5_full

elapsed = time.perf_counter() - t0

datasets = {
    'setup_1': data1, 'setup_2': data2, 'setup_3': data3,
    'setup_4': data4, 'setup_5': data5,
}

# Target censoring rates from synthetic_data_guideline.md
target_censoring = {
    'setup_1': '30-40%', 'setup_2': '40-50%', 'setup_3': '30-50%',
    'setup_4': '30-40%', 'setup_5': '60-70%', 'interp': '~30%',
}

print(f'All datasets generated in {elapsed:.1f}s\n')
print(f'{"Setup":<11s}  {"n":>5s}  {"p":>4s}  {"Events":>6s}  {"Cens.":>7s}  {"Target":>8s}')
print('-' * 55)
for name, d in {**datasets, 'interp': data_interp}.items():
    cens_rate = 1 - d.E.mean()
    target = target_censoring.get(name, '?')
    print(f'{name:11s}  {d.X.shape[0]:5d}  {d.X.shape[1]:4d}  {int(d.E.sum()):6d}  '
          f'{cens_rate:6.1%}  {target:>8s}')

In [ ]:
# ── Kaplan-Meier Curves (2x3 grid) ──
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
all_data = [('Setup 1 — Clean PH', data1), ('Setup 2 — Non-PH', data2),
            ('Setup 3 — High-dim', data3), ('Setup 4 — Competing', data4),
            ('Setup 5 — Large-n', data5), ('Setup Interp', data_interp)]

for ax, (title, d) in zip(axes.flat, all_data):
    kmf = KaplanMeierFitter()
    kmf.fit(d.T, event_observed=d.E)
    kmf.plot_survival_function(ax=ax, ci_show=True)
    cens = 1 - d.E.mean()
    ax.set_title(f'{title}\n(censoring: {cens:.0%})', fontsize=11)
    ax.set_xlabel('Time')
    ax.set_ylabel('S(t)')
    ax.get_legend().remove()

fig.suptitle('Kaplan-Meier Survival Curves', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Event Time Distributions ──
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, (title, d) in zip(axes.flat, all_data):
    events = d.T[d.E == 1]
    censored = d.T[d.E == 0]
    ax.hist(events, bins=40, alpha=0.7, label='Events', color='steelblue')
    ax.hist(censored, bins=40, alpha=0.5, label='Censored', color='orange')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Time')
    ax.legend(fontsize=9)

fig.suptitle('Observed Time Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Heatmaps: Setup 1 (AR(1)) and Setup 3 (block) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Setup 1: AR(1) correlation structure
corr1 = np.corrcoef(data1.X.T)
sns.heatmap(corr1, ax=axes[0], cmap='coolwarm', center=0, vmin=-1, vmax=1,
            xticklabels=data1.feature_names, yticklabels=data1.feature_names)
axes[0].set_title('Setup 1 — AR(1) Correlation', fontsize=11)

# Setup 3: show first 30 features to see block structure
n_show = min(30, data3.X.shape[1])
corr3 = np.corrcoef(data3.X[:, :n_show].T)
sns.heatmap(corr3, ax=axes[1], cmap='coolwarm', center=0, vmin=-1, vmax=1,
            xticklabels=False, yticklabels=False)
axes[1].set_title(f'Setup 3 — Block Correlation (first {n_show} features)', fontsize=11)

plt.tight_layout()
plt.show()

### PH Violation in Setup 2

Setup 2 has a time-varying coefficient β(t) = 1.5 − 0.3t on `X_cont_1`, deliberately violating the proportional hazards assumption. The Schoenfeld residual test should reject PH for this covariate.

In [ ]:
# ── Schoenfeld Residuals Test — Setup 2 ──
df2 = data2.to_dataframe()
cph2 = CoxPHFitter()
cph2.fit(df2, duration_col='T', event_col='E')

print('Schoenfeld residuals test for PH assumption (Setup 2):')
print('Expecting REJECTION for X_cont_1 (time-varying effect)\n')
cph2.check_assumptions(df2, p_value_threshold=0.05, show_plots=False)

In [ ]:
# ── Setup 4: Competing Risks — Cumulative Incidence Functions ──
from lifelines import AalenJohansenFitter

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Cause distribution bar chart
events4 = data4.cause[data4.E == 1]
cause_counts = pd.Series(events4).value_counts().sort_index()
cause_labels = {1: 'Relapse', 2: 'Direct Death'}
axes[0].bar([cause_labels.get(c, f'Cause {c}') for c in cause_counts.index],
            cause_counts.values, color=['steelblue', 'coral'])
axes[0].set_title('Setup 4 — Event Counts by Cause', fontsize=11)
axes[0].set_ylabel('Count')
n_censored = int((data4.E == 0).sum())
axes[0].annotate(f'Censored: {n_censored}', xy=(0.95, 0.95),
                 xycoords='axes fraction', ha='right', va='top', fontsize=10)

# (b) Cumulative Incidence Functions (Aalen-Johansen estimator)
# Proper CIF accounts for competing risks — unlike 1-KM which overestimates
for cause_val, label, color in [(1, 'CIF: Relapse', 'steelblue'), (2, 'CIF: Direct Death', 'coral')]:
    ajf = AalenJohansenFitter(calculate_variance=False)
    # AalenJohansenFitter needs integer event types: 0=censored, 1,2=causes
    event_of_interest = cause_val
    ajf.fit(data4.T, data4.cause, event_of_interest=event_of_interest)
    ajf.plot(ax=axes[1], label=label, color=color)

axes[1].set_title('Setup 4 — Cumulative Incidence Functions (Aalen-Johansen)', fontsize=11)
axes[1].set_xlabel('Time')
axes[1].set_ylabel('CIF(t)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Setup Interp: Ground Truth Effects ──
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

X_interp = data_interp.X
T_interp = data_interp.T
names = data_interp.feature_names

# x0: Linear effect (beta=0.8)
axes[0, 0].scatter(X_interp[:, 0], T_interp, alpha=0.3, s=10, c='steelblue')
axes[0, 0].set_xlabel(names[0])
axes[0, 0].set_ylabel('Event time')
axes[0, 0].set_title('x0: Linear effect (β=0.8)', fontsize=11)

# x1: Threshold effect (1.0 * I(x1 > 0))
group_lo = T_interp[X_interp[:, 1] <= 0]
group_hi = T_interp[X_interp[:, 1] > 0]
axes[0, 1].boxplot([group_lo, group_hi], labels=['x1 ≤ 0', 'x1 > 0'])
axes[0, 1].set_ylabel('Event time')
axes[0, 1].set_title('x1: Threshold effect (I(x1>0))', fontsize=11)

# x2: Quadratic effect (0.5 * x2^2)
x2_sorted = np.sort(X_interp[:, 2])
axes[1, 0].scatter(X_interp[:, 2], T_interp, alpha=0.3, s=10, c='steelblue')
axes[1, 0].set_xlabel(names[2])
axes[1, 0].set_ylabel('Event time')
axes[1, 0].set_title('x2: Quadratic effect (0.5·x²)', fontsize=11)

# x3: Time-varying effect (beta(t) = 0.5 + 0.3t)
t_grid = np.linspace(0, 10, 100)
beta_t = 0.5 + 0.3 * t_grid
axes[1, 1].plot(t_grid, beta_t, linewidth=2, color='steelblue')
axes[1, 1].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='β₀=0.5')
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('β(t)')
axes[1, 1].set_title('x3: Time-varying β(t) = 0.5 + 0.3t', fontsize=11)
axes[1, 1].legend()

fig.suptitle('Setup Interp — Ground Truth Effects', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 2. Model Overview

Six survival models spanning classical statistics to deep learning:

| Model | Class | Library | Assumption | Competing Risks |
|-------|-------|---------|------------|----------------|
| CoxPH | `CoxPHModel` | scikit-survival | PH | Cause-specific |
| CoxNet | `CoxNetModel` | scikit-survival | PH + sparsity | Cause-specific |
| RSF | `RSFModel` | scikit-survival | None (tree) | Cause-specific |
| DeepSurv | `DeepSurvModel` | pycox | PH + nonlinear | Cause-specific |
| DeepHit | `DeepHitModel` | pycox | None | Native |
| SurvTRACE | `SurvTRACEModel` | vendored | None (transformer) | Native |

In [ ]:
# ── Instantiate all models ──
print('Available models:')
for name, cls in ALL_MODELS.items():
    cr = '(competing risks)' if cls.supports_competing_risks else ''
    print(f'  {name:15s}  {cls.__name__:20s}  {cr}')

### Model × Setup Compatibility

| Model | S1 | S2 | S3 | S4 | S5 |
|-------|----|----|----|----|----|
| CoxPH | ✓ | ✓ | ✗ (p>>n) | cause-specific | ✓ |
| CoxNet | ✓ | ✓ | ✓ | cause-specific | ✓ |
| RSF | ✓ | ✓ | ✓ | cause-specific | ✓ |
| DeepSurv | ✓ | ✓ | ✓ | cause-specific | ✓ |
| DeepHit | ✓ | ✓ | ✓ | native | ✓ |
| SurvTRACE | ✓ | ✓ | ✓ | native | ✓ |

---
## 3. Quick Benchmark

Using `BenchmarkRunner` with a simple 80/20 train/test split and default hyperparameters. This serves as a fast smoke test before the full nested CV evaluation.

In [ ]:
# ── BenchmarkRunner: All models × Setups 1-4 ──
all_model_instances = [cls() for cls in ALL_MODELS.values()]

runner_14 = BenchmarkRunner(
    models=all_model_instances,
    setups=['setup_1', 'setup_2', 'setup_3', 'setup_4'],
    seed=SEED,
)
results_14 = runner_14.run()
df_14 = results_14.to_dataframe()
print(f'\nBenchmark Setups 1-4 complete: {len(df_14)} results')
df_14.head(10)

In [ ]:
# ── BenchmarkRunner: Setup 5 — ALL models (scalability test) ──
# Setup 5 tests scalability; all models should be compared per the guideline
s5_model_instances = [cls() for cls in ALL_MODELS.values()]

runner_5 = BenchmarkRunner(
    models=s5_model_instances,
    setups=['setup_5'],
    seed=SEED,
)
results_5 = runner_5.run()
df_5 = results_5.to_dataframe()
print(f'\nSetup 5 benchmark complete: {len(df_5)} results')
df_5

In [ ]:
# ── Merged Results Table ──
df_all_bench = pd.concat([df_14, df_5], ignore_index=True)

# Display styled table
display_cols = ['setup', 'model', 'c_index', 'c_index_ipcw', 'ibs', 'fit_time_s']
available_cols = [c for c in display_cols if c in df_all_bench.columns]
styled = df_all_bench[available_cols].style.format({
    'c_index': '{:.4f}', 'c_index_ipcw': '{:.4f}',
    'ibs': '{:.4f}', 'fit_time_s': '{:.1f}',
}).background_gradient(subset=['c_index'], cmap='Greens')
styled

In [ ]:
# ── C-index and IBS Heatmaps (from BenchmarkRunner) ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, metric, title, cmap, fmt in [
    (axes[0], 'c_index', 'Harrell C-index (↑)', 'Greens', '.3f'),
    (axes[1], 'ibs', 'Integrated Brier Score (↓)', 'Reds_r', '.4f'),
]:
    if metric not in df_all_bench.columns:
        ax.set_visible(False)
        continue
    pivot = df_all_bench.pivot_table(index='model', columns='setup', values=metric)
    sns.heatmap(pivot, annot=True, fmt=fmt, cmap=cmap, ax=ax, linewidths=0.5)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('')
    ax.set_ylabel('')

fig.suptitle('Quick Benchmark Results', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Fit Time Bar Chart ──
if 'fit_time_s' in df_all_bench.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    pivot_time = df_all_bench.pivot_table(index='model', columns='setup', values='fit_time_s')
    pivot_time.plot(kind='bar', ax=ax, width=0.8)
    ax.set_ylabel('Fit time (seconds)')
    ax.set_title('Training Time by Model and Setup', fontsize=12)
    ax.legend(title='Setup', bbox_to_anchor=(1.02, 1), loc='upper left')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Brier Score Over Time — Setups 1 and 2 ──
# Show time-dependent calibration on both a well-specified (S1) and mis-specified (S2) scenario

eval_setups = [('Setup 1', data1), ('Setup 2', data2)]
brier_models_names = ['cox_ph', 'rsf', 'deep_surv']

# Store fitted models and splits for reuse in calibration + AUC cells
fitted_models_per_setup = {}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (setup_label, data) in zip(axes, eval_setups):
    X_tr, X_te, T_tr, T_te, E_tr, E_te = train_test_split(
        data.X, data.T, data.E, test_size=0.2, random_state=SEED, stratify=data.E
    )
    y_train = Surv.from_arrays(E_tr.astype(bool), T_tr)
    y_test = Surv.from_arrays(E_te.astype(bool), T_te)

    t_min = np.percentile(T_te[E_te == 1], 5)
    t_max = np.percentile(T_te[E_te == 1], 95)
    eval_times = np.linspace(t_min, t_max, 15)

    brier_dict = {}
    setup_models = {}
    for mname in brier_models_names:
        tag = f'bench_{setup_label.lower().replace(" ", "_")}'
        if RETRAIN_MODELS or not model_exists(mname, tag):
            model = ALL_MODELS[mname]()
            model.fit(X_tr, T_tr, E_tr)
            save_model(model, mname, tag)
        else:
            model = load_model(mname, tag)
        setup_models[mname] = model

        sf = model.predict_survival_function(X_te, eval_times)
        try:
            _, bs = brier_score(y_train, y_test, sf, eval_times)
            brier_dict[mname] = bs
        except Exception as e:
            print(f'  Brier failed for {mname} on {setup_label}: {e}')

    fitted_models_per_setup[setup_label] = {
        'models': setup_models,
        'X_tr': X_tr, 'X_te': X_te, 'T_tr': T_tr, 'T_te': T_te,
        'E_tr': E_tr, 'E_te': E_te, 'eval_times': eval_times,
    }

    plot_brier_score_over_time(eval_times, brier_dict, setup_name=setup_label, ax=ax)

fig.suptitle('Time-Dependent Brier Score', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Calibration Curves — Setups 1, 2, and 5 ──
# Show calibration across well-specified (S1), non-PH (S2), and heavy censoring (S5)

# Add Setup 5 split if not already available
if 'Setup 5' not in fitted_models_per_setup:
    X5_tr, X5_te, T5_tr, T5_te, E5_tr, E5_te = train_test_split(
        data5.X, data5.T, data5.E, test_size=0.2, random_state=SEED, stratify=data5.E
    )
    s5_models = {}
    for mname in ['cox_net', 'rsf', 'deep_surv']:
        tag = 'bench_setup_5'
        if RETRAIN_MODELS or not model_exists(mname, tag):
            model = ALL_MODELS[mname]()
            model.fit(X5_tr, T5_tr, E5_tr)
            save_model(model, mname, tag)
        else:
            model = load_model(mname, tag)
        s5_models[mname] = model
    t5_min = np.percentile(T5_te[E5_te == 1], 5)
    t5_max = np.percentile(T5_te[E5_te == 1], 95)
    fitted_models_per_setup['Setup 5'] = {
        'models': s5_models,
        'X_tr': X5_tr, 'X_te': X5_te, 'T_tr': T5_tr, 'T_te': T5_te,
        'E_tr': E5_tr, 'E_te': E5_te, 'eval_times': np.linspace(t5_min, t5_max, 15),
    }

calib_setups = ['Setup 1', 'Setup 2', 'Setup 5']
fig, axes = plt.subplots(1, len(calib_setups), figsize=(6 * len(calib_setups), 6))

for ax, setup_label in zip(axes, calib_setups):
    info = fitted_models_per_setup[setup_label]
    t_median = np.median(info['T_te'][info['E_te'] == 1])
    calib_data = []
    for mname, model in info['models'].items():
        try:
            sf_at_t = model.predict_survival_function(info['X_te'], np.array([t_median]))[:, 0]
            pred_means, obs_surv = calibration_curve(
                info['T_te'], info['E_te'], sf_at_t, t_median, n_bins=10
            )
            calib_data.append((mname, pred_means, obs_surv))
        except Exception as e:
            print(f'  Calibration failed for {mname} on {setup_label}: {e}')
    if calib_data:
        plot_calibration_comparison(calib_data, setup_name=f'{setup_label} (t={t_median:.1f})', ax=ax)

fig.suptitle('Calibration Curves', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Time-Dependent AUC — Setup 1 (PH) and Setup 2 (Non-PH) ──
# Comparing CoxPH vs RSF across both scenarios to show where PH breaks down

auc_model_names = ['cox_ph', 'rsf']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, setup_label in zip(axes, ['Setup 1', 'Setup 2']):
    info = fitted_models_per_setup[setup_label]
    t_min = np.percentile(info['T_te'][info['E_te'] == 1], 10)
    t_max = np.percentile(info['T_te'][info['E_te'] == 1], 90)
    auc_eval_times = np.linspace(t_min, t_max, 15)

    for mname in auc_model_names:
        model = info['models'][mname]
        risk = model.predict_risk(info['X_te'])
        try:
            auc_values, mean_auc = time_dependent_auc(
                info['T_tr'], info['E_tr'], info['T_te'], info['E_te'],
                risk, auc_eval_times
            )
            ax.plot(auc_eval_times, auc_values,
                    label=f'{mname} (mean AUC={mean_auc:.3f})', linewidth=2)
        except Exception as e:
            print(f'  AUC failed for {mname} on {setup_label}: {e}')

    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
    ax.set_xlabel('Time')
    ax.set_ylabel('AUC(t)')
    ax.set_title(f'Time-Dependent AUC — {setup_label}', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('Time-Dependent AUC: CoxPH vs RSF', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Full Evaluation — Nested Cross-Validation

Nested 5×5 cross-validation with hyperparameter tuning via random search.
- **Outer loop**: 5-fold stratified CV for unbiased evaluation
- **Inner loop**: 5-fold CV for HP selection

In `QUICK_MODE`, we run only 2 setups × 3 classical models to keep runtime manageable.

In [ ]:
# ── Nested CV ──
if QUICK_MODE:
    cv_setup_names = ['setup_1', 'setup_2']
    cv_model_names = ['cox_ph', 'cox_net', 'rsf']
else:
    cv_setup_names = ['setup_1', 'setup_2', 'setup_3', 'setup_4', 'setup_5']
    cv_model_names = list(ALL_MODELS.keys())

print(f'Running NestedCV: {cv_setup_names} x {cv_model_names}')
print('This may take 15-30 minutes...\n')

cv_runner = NestedCVRunner(
    model_names=cv_model_names,
    setups=cv_setup_names,
    n_outer_folds=5,
    n_inner_folds=5,
    seed=SEED,
)
cv_results = cv_runner.run()
print('\nNested CV complete!')

In [ ]:
# ── Full run configuration (for complete experiment) ──
# To run the full experiment with all models and setups, set QUICK_MODE = False
# at the top of this notebook. The cell above will then use:
#   cv_setup_names = ['setup_1', 'setup_2', 'setup_3', 'setup_4', 'setup_5']
#   cv_model_names = ['cox_ph', 'cox_net', 'rsf', 'deep_surv', 'deep_hit', 'survtrace']
#
# For multi-repeat experiments (recommended for final results):
#
#   cv_runner_full = NestedCVRunner(
#       model_names=list(ALL_MODELS.keys()),
#       setups=['setup_1', 'setup_2', 'setup_3', 'setup_4', 'setup_5'],
#       n_outer_folds=5,
#       n_inner_folds=5,
#       n_repeats=3,
#       seed=SEED,
#   )
#   cv_results_full = cv_runner_full.run()

if QUICK_MODE:
    print('QUICK_MODE active — running partial nested CV (2 setups x 3 models).')
    print('Set QUICK_MODE = False for the complete experiment.')
else:
    print('Full nested CV complete — all 5 setups x 6 models evaluated.')

In [ ]:
# ── Metric Heatmaps (from Nested CV) ──
agg = cv_results.aggregate()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

try:
    plot_metric_heatmap(agg, 'c_index_ipcw', ax=axes[0])
    axes[0].set_title('IPCW C-index (mean ± SD)', fontsize=12)
except Exception as e:
    print(f'C-index heatmap error: {e}')

try:
    plot_metric_heatmap(agg, 'ibs', ax=axes[1])
    axes[1].set_title('Integrated Brier Score (mean ± SD)', fontsize=12)
except Exception as e:
    print(f'IBS heatmap error: {e}')

fig.suptitle('Nested CV Results', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Formatted Table + LaTeX ──
fmt_table = cv_results.format_table()
display(fmt_table)

print('\n--- LaTeX output ---')
print(fmt_table.to_latex())

In [ ]:
# ── Rank Chart ──
rank_df = cv_results.rank_table('c_index_ipcw')
fig = plot_rank_chart(rank_df)
plt.show()

print('\nModel rankings (IPCW C-index):')
display(rank_df)

---
## 5. Per-Setup Deep Dives

In [ ]:
# ── Setup 1: True vs Estimated Betas ──
df1 = data1.to_dataframe()
cph1 = CoxPHFitter()
cph1.fit(df1, duration_col='T', event_col='E')

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(data1.feature_names))
width = 0.35

# true_betas is a numpy array for Setup 1
true_betas = np.asarray(data1.true_betas).flatten()
est_betas = np.array([cph1.params_[name] for name in data1.feature_names])

ax.bar(x_pos - width/2, true_betas, width, label='True β', color='steelblue', alpha=0.8)
ax.bar(x_pos + width/2, est_betas, width, label='Estimated β', color='coral', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(data1.feature_names, rotation=45, ha='right')
ax.set_ylabel('Coefficient')
ax.set_title('Setup 1 — True vs CoxPH Estimated Betas', fontsize=12)
ax.legend()
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

# Print numerical comparison
beta_df = pd.DataFrame({
    'Feature': data1.feature_names,
    'True β': true_betas,
    'Estimated β': est_betas,
    'Absolute Error': np.abs(true_betas - est_betas),
})
print(f'Mean absolute error: {beta_df["Absolute Error"].mean():.4f}')
display(beta_df)

In [ ]:
# ── Setup 2: Model Comparison Under Non-PH ──
s2_results = df_all_bench[df_all_bench['setup'] == 'setup_2'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'c_index' in s2_results.columns:
    colors = ['coral' if m == 'cox_ph' else 'steelblue' for m in s2_results['model']]
    axes[0].bar(s2_results['model'], s2_results['c_index'], color=colors)
    axes[0].set_ylabel('Harrell C-index')
    axes[0].set_title('Setup 2 — C-index (CoxPH misspecified)', fontsize=11)
    axes[0].set_xticklabels(s2_results['model'], rotation=45, ha='right')

if 'ibs' in s2_results.columns:
    colors = ['coral' if m == 'cox_ph' else 'steelblue' for m in s2_results['model']]
    axes[1].bar(s2_results['model'], s2_results['ibs'], color=colors)
    axes[1].set_ylabel('IBS (lower is better)')
    axes[1].set_title('Setup 2 — IBS (CoxPH misspecified)', fontsize=11)
    axes[1].set_xticklabels(s2_results['model'], rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# ── Setup 3: CoxNet Sparsity Analysis ──
from src.models.cox_net import CoxNetModel

# Fit CoxNet on Setup 3
coxnet = CoxNetModel()
coxnet.fit(data3.X, data3.T, data3.E)

# Get coefficients
coefs = coxnet._model.coef_.flatten() if hasattr(coxnet._model, 'coef_') else np.zeros(data3.X.shape[1])
nonzero = np.nonzero(coefs)[0]

# Active indices from true_betas dict (first feature in each of 20 blocks: 0, 100, ..., 1900)
active = set(data3.true_betas['active_indices'])

fig, ax = plt.subplots(figsize=(12, 4))
n_show = min(50, len(coefs))
colors = []
for i in range(n_show):
    if i in active and coefs[i] != 0:
        colors.append('steelblue')   # True positive
    elif i in active:
        colors.append('lightblue')   # Missed (false negative)
    elif coefs[i] != 0:
        colors.append('coral')       # False positive
    else:
        colors.append('lightgray')   # True negative

ax.bar(range(n_show), np.abs(coefs[:n_show]), color=colors, width=1.0)
ax.set_xlabel('Feature index')
ax.set_ylabel('|Coefficient|')
ax.set_title(f'Setup 3 — CoxNet Coefficients (first {n_show} of {len(coefs)}, '
             f'{len(nonzero)} non-zero)', fontsize=11)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label='True positive'),
    Patch(facecolor='lightblue', label='False negative'),
    Patch(facecolor='coral', label='False positive'),
    Patch(facecolor='lightgray', label='True negative'),
]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.show()

# Report recovery statistics
recovered = active & set(nonzero)
precision = len(recovered) / max(len(nonzero), 1)
recall = len(recovered) / len(active)
print(f'Active features (20 true): {sorted(active)[:10]}... (showing first 10)')
print(f'Non-zero CoxNet features:  {len(nonzero)}')
print(f'Recovered (TP):            {len(recovered)}')
print(f'Precision:                 {precision:.2%}')
print(f'Recall:                    {recall:.2%}')

In [ ]:
# ── Setup 5: Harrell vs IPCW C-index ──
s5_results = df_all_bench[df_all_bench['setup'] == 'setup_5'].copy()

if len(s5_results) > 0 and 'c_index' in s5_results.columns and 'c_index_ipcw' in s5_results.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    x_pos = np.arange(len(s5_results))
    width = 0.35

    ax.bar(x_pos - width/2, s5_results['c_index'].values, width,
           label='Harrell C-index', color='steelblue', alpha=0.8)
    ax.bar(x_pos + width/2, s5_results['c_index_ipcw'].values, width,
           label='IPCW C-index', color='coral', alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(s5_results['model'].values, rotation=45, ha='right')
    ax.set_ylabel('C-index')
    ax.set_title('Setup 5 — Harrell vs IPCW C-index (64% censoring)', fontsize=12)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Setup 5 results not available for comparison.')

---
## 6. Cross-Setup Summary

In [ ]:
# ── Learning Curves: Setup 1 at varying sample sizes ──
sample_sizes = [100, 250, 500, 800]
test_size = 200
lc_models = {'cox_ph': ALL_MODELS['cox_ph'], 'rsf': ALL_MODELS['rsf']}
lc_results = {mname: [] for mname in lc_models}

for n_train in sample_sizes:
    total_needed = n_train + test_size
    if total_needed > data1.X.shape[0]:
        break
    rng = np.random.default_rng(SEED)
    idx = rng.choice(data1.X.shape[0], total_needed, replace=False)
    X_sub, T_sub, E_sub = data1.X[idx], data1.T[idx], data1.E[idx]

    # Ensure enough events for stratification
    n_events = int(E_sub.sum())
    can_stratify = n_events >= 2 and (len(E_sub) - n_events) >= 2
    strat = E_sub if can_stratify else None

    X_tr, X_te, T_tr, T_te, E_tr, E_te = train_test_split(
        X_sub, T_sub, E_sub, test_size=test_size, random_state=SEED, stratify=strat
    )

    for mname, mcls in lc_models.items():
        model = mcls()
        model.fit(X_tr, T_tr, E_tr)
        metrics = evaluate_model(model, X_tr, T_tr, E_tr, X_te, T_te, E_te)
        lc_results[mname].append(metrics.get('c_index', np.nan))

fig, ax = plt.subplots(figsize=(8, 5))
for mname, scores in lc_results.items():
    ax.plot(sample_sizes[:len(scores)], scores, 'o-', label=mname, linewidth=2, markersize=8)

ax.set_xlabel('Training sample size')
ax.set_ylabel('Harrell C-index')
ax.set_title('Learning Curves — Setup 1', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Overall Pivot Table ──
pivot_cindex = df_all_bench.pivot_table(
    index='model', columns='setup', values='c_index'
).round(4)

# Add mean column
pivot_cindex['mean'] = pivot_cindex.mean(axis=1)
pivot_cindex = pivot_cindex.sort_values('mean', ascending=False)

styled_pivot = pivot_cindex.style.background_gradient(
    cmap='Greens', axis=None
).format('{:.4f}')

print('Overall C-index Pivot Table (Benchmark):')
styled_pivot

---
## 7. Interpretability

Using **Setup Interp** with known ground-truth effects to evaluate SurvLIME and SurvSHAP(t).

**Ground truth (4 important features):**
- `X_cont_0`: Linear effect (β=0.8)
- `X_cont_1`: Threshold effect (1.0 · I(x₁>0))
- `X_cont_2`: Quadratic effect (0.5 · x₂²)
- `X_cont_3`: Time-varying effect (β(t) = 0.5 + 0.3t)
- `X_cont_4` – `X_cont_7`, `X_bin_0` – `X_bin_3`: Noise (zero effect)

In [ ]:
# ── Train/Test Split for Interpretability ──
X_int_tr, X_int_te, T_int_tr, T_int_te, E_int_tr, E_int_te = train_test_split(
    data_interp.X, data_interp.T, data_interp.E,
    test_size=0.2, random_state=SEED, stratify=data_interp.E
)

# Time grid: 50 points (guideline recommends 50-100) clipped to 95th percentile
event_times = T_int_tr[E_int_tr == 1]
t_p95 = np.percentile(event_times, 95)
interp_times = np.linspace(event_times.min(), t_p95, 50)

feature_names = data_interp.feature_names
true_important = ['X_cont_0', 'X_cont_1', 'X_cont_2', 'X_cont_3']

# Ground-truth effect signs (for sign consistency checks)
# x0: linear +0.8 (risk-increasing), x1: threshold +1.0, x2: quadratic +0.5, x3: time-varying +0.5+0.3t
true_signs = {'X_cont_0': +1, 'X_cont_1': +1, 'X_cont_2': +1, 'X_cont_3': +1}

print(f'Training: {X_int_tr.shape[0]} samples, Test: {X_int_te.shape[0]} samples')
print(f'Time grid: {interp_times.min():.2f} to {interp_times.max():.2f} ({len(interp_times)} points)')
print(f'Features: {feature_names}')
print(f'True important: {true_important}')

In [ ]:
# ── Fit CoxPH + RSF on Setup Interp ──
interp_models = {}

for mname in ['cox_ph', 'rsf']:
    tag = 'interp'
    if RETRAIN_MODELS or not model_exists(mname, tag):
        model = ALL_MODELS[mname]()
        model.fit(X_int_tr, T_int_tr, E_int_tr)
        save_model(model, mname, tag)
    else:
        model = load_model(mname, tag)
    interp_models[mname] = model

    metrics = evaluate_model(model, X_int_tr, T_int_tr, E_int_tr,
                            X_int_te, T_int_te, E_int_te)
    print(f'{mname}: C-index={metrics.get("c_index", "N/A"):.4f}, '
          f'IPCW C-index={metrics.get("c_index_ipcw", "N/A"):.4f}, '
          f'IBS={metrics.get("ibs", "N/A"):.4f}')

In [ ]:
# ── Select 3 Individuals by Risk Percentile (P10, P50, P90) ──
risk_scores = interp_models['rsf'].predict_risk(X_int_te)
percentiles = [10, 50, 90]
individual_indices = []

for p in percentiles:
    threshold = np.percentile(risk_scores, p)
    idx = np.argmin(np.abs(risk_scores - threshold))
    individual_indices.append(idx)
    print(f'P{p}: index={idx}, risk={risk_scores[idx]:.4f}')

print(f'\nSelected individuals: {individual_indices}')

### 7.1 SurvLIME

SurvLIME fits a local Cox surrogate model around each individual, producing feature-level importance coefficients.

In [ ]:
# ── SurvLIME — CoxPH: bar plots + surrogate curve overlay ──
import logging
logging.disable(logging.WARNING)

from src.interpretability import SurvLIMEExplainer

lime_cox = SurvLIMEExplainer(
    model=interp_models['cox_ph'],
    X_train=X_int_tr, T_train=T_int_tr, E_train=E_int_tr,
)

# Bar plots for 3 representative individuals
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
lime_coefs_cox = {}
for ax, idx, pct in zip(axes, individual_indices, percentiles):
    coefs = lime_cox.explain(X_int_te[idx])
    lime_coefs_cox[idx] = coefs
    lime_cox.plot_weights(coefs, feature_names, true_important=true_important, ax=ax)
    ax.set_title(f'CoxPH — P{pct} individual', fontsize=11)

fig.suptitle('SurvLIME Local Explanations — CoxPH', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Black-box vs SurvLIME surrogate survival curve overlay (guideline requirement)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (idx, pct) in zip(axes, [(individual_indices[0], percentiles[0]),
                                   (individual_indices[2], percentiles[2])]):
    lime_cox.compare_survival_curves(
        X_int_te[idx], interp_times,
        coefficients=lime_coefs_cox[idx], ax=ax,
    )
    ax.set_title(f'CoxPH — P{pct} individual', fontsize=11)

fig.suptitle('SurvLIME: Black-box vs Surrogate Survival Curves', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── SurvLIME — RSF: bar plots + surrogate curve overlay ──
lime_rsf = SurvLIMEExplainer(
    model=interp_models['rsf'],
    X_train=X_int_tr, T_train=T_int_tr, E_train=E_int_tr,
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
lime_coefs_rsf = {}
for ax, idx, pct in zip(axes, individual_indices, percentiles):
    coefs = lime_rsf.explain(X_int_te[idx])
    lime_coefs_rsf[idx] = coefs
    lime_rsf.plot_weights(coefs, feature_names, true_important=true_important, ax=ax)
    ax.set_title(f'RSF — P{pct} individual', fontsize=11)

fig.suptitle('SurvLIME Local Explanations — RSF', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Surrogate curve overlay for RSF (low-risk and high-risk)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (idx, pct) in zip(axes, [(individual_indices[0], percentiles[0]),
                                   (individual_indices[2], percentiles[2])]):
    lime_rsf.compare_survival_curves(
        X_int_te[idx], interp_times,
        coefficients=lime_coefs_rsf[idx], ax=ax,
    )
    ax.set_title(f'RSF — P{pct} individual', fontsize=11)

fig.suptitle('SurvLIME: Black-box (RSF) vs Surrogate Survival Curves', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── SurvLIME Precision@4, Sign Consistency ──
lime_summary = []

for mname, lime_expl, coefs_dict in [('cox_ph', lime_cox, lime_coefs_cox),
                                       ('rsf', lime_rsf, lime_coefs_rsf)]:
    precisions = []
    sign_matches = []
    for idx in individual_indices:
        coefs = coefs_dict[idx]
        # Top-4 by absolute value
        top4_idx = np.argsort(np.abs(coefs))[-4:]
        top4_names = {feature_names[i] for i in top4_idx}
        precision = len(top4_names & set(true_important)) / 4
        precisions.append(precision)

        # Sign consistency for important features
        for fname, expected_sign in true_signs.items():
            fi = feature_names.index(fname)
            if coefs[fi] != 0:
                actual_sign = np.sign(coefs[fi])
                sign_matches.append(actual_sign == expected_sign)

    lime_summary.append({
        'model': mname,
        'method': 'SurvLIME',
        'precision@4_mean': np.mean(precisions),
        'precision@4_std': np.std(precisions),
        'sign_consistency': np.mean(sign_matches) if sign_matches else np.nan,
    })

lime_df = pd.DataFrame(lime_summary)
print('SurvLIME Precision@4 and Sign Consistency:')
display(lime_df)

### 7.2 SurvSHAP(t)

SurvSHAP(t) computes time-dependent Shapley values, capturing how feature importance varies over time.

In [ ]:
# ── SurvSHAP(t) — RSF High-Risk Individual ──
from src.interpretability import SurvSHAPExplainer

shap_rsf = SurvSHAPExplainer(
    model=interp_models['rsf'],
    X_train=X_int_tr, T_train=T_int_tr, E_train=E_int_tr,
    feature_names=feature_names,
)

# High-risk individual (P90)
high_risk_idx = individual_indices[2]
shap_result_hi = shap_rsf.explain(X_int_te[high_risk_idx], interp_times)

fig = shap_rsf.plot_shap_over_time(
    shap_result_hi, feature_names,
    highlight_features=true_important,
)
plt.suptitle('SurvSHAP(t) — RSF, High-Risk Individual (P90)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
# ── SurvSHAP(t): Low vs High Risk Comparison ──
low_risk_idx = individual_indices[0]
shap_result_lo = shap_rsf.explain(X_int_te[low_risk_idx], interp_times)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

shap_rsf.plot_shap_over_time(
    shap_result_lo, feature_names,
    highlight_features=true_important, ax=axes[0],
)
axes[0].set_title('Low-Risk (P10)', fontsize=11)

shap_rsf.plot_shap_over_time(
    shap_result_hi, feature_names,
    highlight_features=true_important, ax=axes[1],
)
axes[1].set_title('High-Risk (P90)', fontsize=11)

fig.suptitle('SurvSHAP(t) — Low vs High Risk Contrast (RSF)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Global SurvSHAP Importance ──
# Use full test set for global importance (guideline: average over test set)
n_global = X_int_te.shape[0]
X_global = X_int_te

print(f'Computing global SurvSHAP importance over {n_global} individuals...')
global_importance = shap_rsf.explain_multiple(X_global, interp_times, feature_names)

fig = shap_rsf.plot_global_importance(
    global_importance, true_important=true_important, top_k=12,
)
plt.suptitle('Global Feature Importance — SurvSHAP(t), RSF', fontsize=12, fontweight='bold')
plt.show()

# Per-individual precision@4 with mean +/- std
shap_precisions_rsf = []
for i in range(n_global):
    result_i = shap_rsf.explain(X_int_te[i], interp_times)
    from src.interpretability.explainers import _shap_time_cols
    shap_cols = _shap_time_cols(result_i)
    if shap_cols:
        shap_vals = result_i[shap_cols].values.astype(float)
        abs_mean_i = np.nanmean(np.abs(shap_vals), axis=1)
        var_names = result_i['variable_name'].values if 'variable_name' in result_i.columns else feature_names
        top4_i = set(var_names[np.argsort(abs_mean_i)[-4:]])
        p_at_4_i = len(top4_i & set(true_important)) / 4
        shap_precisions_rsf.append(p_at_4_i)

print(f'\nGlobal Precision@4 (from aggregated importance): '
      f'{len(set(global_importance.head(4)["feature"]) & set(true_important)) / 4:.2f}')
print(f'Per-individual Precision@4: {np.mean(shap_precisions_rsf):.2f} +/- {np.std(shap_precisions_rsf):.2f}')
print(f'Top-4 global features: {set(global_importance.head(4)["feature"])}')
print(f'True important:        {set(true_important)}')

In [ ]:
# ── CoxPH vs RSF SurvSHAP(t) Comparison ──
shap_cox = SurvSHAPExplainer(
    model=interp_models['cox_ph'],
    X_train=X_int_tr, T_train=T_int_tr, E_train=E_int_tr,
    feature_names=feature_names,
)

# Compare on high-risk individual
shap_result_cox_hi = shap_cox.explain(X_int_te[high_risk_idx], interp_times)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

shap_cox.plot_shap_over_time(
    shap_result_cox_hi, feature_names,
    highlight_features=true_important, ax=axes[0],
)
axes[0].set_title('CoxPH — SurvSHAP(t)', fontsize=11)

shap_rsf.plot_shap_over_time(
    shap_result_hi, feature_names,
    highlight_features=true_important, ax=axes[1],
)
axes[1].set_title('RSF — SurvSHAP(t)', fontsize=11)

fig.suptitle('CoxPH vs RSF — SurvSHAP(t), High-Risk Individual',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Interpretability Summary Table ──
# Global SurvSHAP for CoxPH
print('Computing CoxPH global SurvSHAP...')
global_importance_cox = shap_cox.explain_multiple(X_global, interp_times, feature_names)
top4_cox = set(global_importance_cox.head(4)['feature'].values)
p_at_4_cox_global = len(top4_cox & set(true_important)) / 4

# Per-individual precision@4 for CoxPH SurvSHAP
shap_precisions_cox = []
for i in range(n_global):
    result_i = shap_cox.explain(X_int_te[i], interp_times)
    shap_cols = _shap_time_cols(result_i)
    if shap_cols:
        shap_vals = result_i[shap_cols].values.astype(float)
        abs_mean_i = np.nanmean(np.abs(shap_vals), axis=1)
        var_names = result_i['variable_name'].values if 'variable_name' in result_i.columns else feature_names
        top4_i = set(var_names[np.argsort(abs_mean_i)[-4:]])
        p_at_4_i = len(top4_i & set(true_important)) / 4
        shap_precisions_cox.append(p_at_4_i)

# Build summary
p_at_4_rsf_global = len(set(global_importance.head(4)['feature'].values) & set(true_important)) / 4

summary_rows = [
    *lime_summary,
    {
        'model': 'cox_ph', 'method': 'SurvSHAP(t)',
        'precision@4_mean': np.mean(shap_precisions_cox),
        'precision@4_std': np.std(shap_precisions_cox),
        'sign_consistency': np.nan,  # SurvSHAP signs are time-varying
    },
    {
        'model': 'rsf', 'method': 'SurvSHAP(t)',
        'precision@4_mean': np.mean(shap_precisions_rsf),
        'precision@4_std': np.std(shap_precisions_rsf),
        'sign_consistency': np.nan,
    },
]

summary_df = pd.DataFrame(summary_rows)
# Format nicely
summary_df['precision@4'] = summary_df.apply(
    lambda r: f"{r['precision@4_mean']:.2f} +/- {r['precision@4_std']:.2f}", axis=1
)
summary_df['sign_consistency'] = summary_df['sign_consistency'].apply(
    lambda x: f'{x:.0%}' if not pd.isna(x) else '—'
)

print('\n=== Interpretability Summary ===')
display(summary_df[['model', 'method', 'precision@4', 'sign_consistency']])

---
## 8. Conclusions

### Key Findings

**Per-Setup:**
- **Setup 1 (Clean PH):** CoxPH recovers true betas accurately — serves as a sanity check
- **Setup 2 (Non-PH):** Flexible models (RSF, DeepHit) outperform CoxPH when PH is violated; time-dependent AUC shows divergence over time
- **Setup 3 (High-dim):** CoxNet's elastic-net regularization handles p >> n; CoxPH fails
- **Setup 4 (Competing):** Native competing-risk models (DeepHit, SurvTRACE) vs cause-specific wrappers; CIF (Aalen-Johansen) provides proper visualization
- **Setup 5 (Large-n):** IPCW C-index is essential under heavy censoring; Harrell's C-index can be misleading; calibration diverges across models

**Interpretability:**
- SurvLIME provides simple feature importance via local Cox surrogates; surrogate curves approximate the black-box well for CoxPH
- SurvSHAP(t) captures time-varying importance — critical for non-PH effects (x3 feature)
- Both methods reliably identify ground-truth important features (Precision@4 reported with uncertainty)
- Sign consistency confirms correct effect direction for SurvLIME

**Evaluation:**
- Multiple metrics (C-index, IPCW C-index, IBS, time-dependent AUC, calibration) provide complementary views
- Nested 5×5 CV with HP tuning ensures unbiased evaluation with uncertainty estimates

In [ ]:
# ── 2x2 Summary Figure ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (0,0) C-index heatmap
if 'c_index' in df_all_bench.columns:
    pivot = df_all_bench.pivot_table(index='model', columns='setup', values='c_index')
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='Greens', ax=axes[0, 0], linewidths=0.5)
    axes[0, 0].set_title('C-index Heatmap (Benchmark)', fontsize=11)
    axes[0, 0].set_xlabel('')

# (0,1) Rank chart (from CV if available)
try:
    plot_rank_chart(rank_df, ax=axes[0, 1])
    axes[0, 1].set_title('Model Rankings (Nested CV)', fontsize=11)
except Exception:
    axes[0, 1].text(0.5, 0.5, 'Rank chart\n(run full CV)', ha='center', va='center',
                    transform=axes[0, 1].transAxes, fontsize=14)
    axes[0, 1].set_title('Model Rankings', fontsize=11)

# (1,0) Global SurvSHAP importance
try:
    shap_rsf.plot_global_importance(
        global_importance, true_important=true_important, top_k=8, ax=axes[1, 0],
    )
    axes[1, 0].set_title('Global Feature Importance (SurvSHAP)', fontsize=11)
except Exception:
    axes[1, 0].text(0.5, 0.5, 'SurvSHAP importance', ha='center', va='center',
                    transform=axes[1, 0].transAxes, fontsize=14)

# (1,1) Brier over time — recompute for Setup 1 from stored models
try:
    info_s1 = fitted_models_per_setup['Setup 1']
    y_tr = Surv.from_arrays(info_s1['E_tr'].astype(bool), info_s1['T_tr'])
    y_te = Surv.from_arrays(info_s1['E_te'].astype(bool), info_s1['T_te'])
    brier_dict_s1 = {}
    for mname, model in info_s1['models'].items():
        sf = model.predict_survival_function(info_s1['X_te'], info_s1['eval_times'])
        try:
            _, bs = brier_score(y_tr, y_te, sf, info_s1['eval_times'])
            brier_dict_s1[mname] = bs
        except Exception:
            pass
    plot_brier_score_over_time(info_s1['eval_times'], brier_dict_s1, setup_name='Setup 1', ax=axes[1, 1])
    axes[1, 1].set_title('Brier Score Over Time (Setup 1)', fontsize=11)
except Exception:
    axes[1, 1].text(0.5, 0.5, 'Brier over time', ha='center', va='center',
                    transform=axes[1, 1].transAxes, fontsize=14)

fig.suptitle('Survival Analysis Benchmark — Summary', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(str(RESULTS_DIR / 'summary_figure.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Summary figure saved to {RESULTS_DIR / "summary_figure.png"}')

In [ ]:
# ── Save All Results ──

# Benchmark results
df_all_bench.to_csv(str(RESULTS_DIR / 'benchmark_results.csv'), index=False)
print(f'Saved: {RESULTS_DIR / "benchmark_results.csv"}')

# CV results (if available)
try:
    cv_df = cv_results.to_dataframe()
    cv_df.to_csv(str(RESULTS_DIR / 'nested_cv_results.csv'), index=False)
    agg.to_csv(str(RESULTS_DIR / 'nested_cv_aggregated.csv'), index=False)
    print(f'Saved: {RESULTS_DIR / "nested_cv_results.csv"}')
    print(f'Saved: {RESULTS_DIR / "nested_cv_aggregated.csv"}')
except Exception:
    print('CV results not available to save.')

# Interpretability summary
summary_df.to_csv(str(RESULTS_DIR / 'interpretability_notebook_summary.csv'), index=False)
print(f'Saved: {RESULTS_DIR / "interpretability_notebook_summary.csv"}')

print(f'\nAll results saved to {RESULTS_DIR}/')
print(f'Saved models in {SAVED_MODELS_DIR}/')